# Aula 2 · Do dado ao modelo (texto + tabela, juntos)

Aqui montamos o modelo **de verdade**: um único modelo que usa o **texto** da avaliação
**e** os dados de **tabela** (preço, frete, prazo) ao mesmo tempo.

Por que juntos? Porque só **37%** dos clientes escrevem um comentário. Um modelo só de
texto seria cego para os outros 63%. Juntando as duas fontes, o modelo:
- usa o texto quando ele existe,
- e se vira com a tabela quando não existe.

Assim ele consegue prever para **todos** os pedidos.

> Pré-requisito: ter visto o notebook `00-como-texto-vira-numero-tfidf` (como o texto vira numero).

## 1. Pegar os dados — TODOS os pedidos

Diferente de antes: nao filtramos so quem tem texto. Pegamos todo mundo.
Quem nao escreveu comentario fica com texto vazio.

In [ ]:
import urllib.request, pathlib
import pandas as pd

URL = "https://github.com/fenakamuta/poliusppro-data-engineering/releases/download/aula1-olist-2026-v1/olist.parquet"
PATH = pathlib.Path("olist.parquet")
if not PATH.exists():
    print("Baixando..."); urllib.request.urlretrieve(URL, PATH)

df = pd.read_parquet(PATH)
print(f"{len(df):,} pedidos no total")

tem = df["review_comment_message"].fillna("").str.len() > 0
print(f"Com comentario:  {tem.sum():,} ({tem.mean()*100:.0f}%)")
print(f"Sem comentario:  {(~tem).sum():,} ({(~tem).mean()*100:.0f}%)")

## 2. Montar features e rotulo

- **texto**: o comentario (vazio quando nao existe)
- **tabela**: preco, frete, prazo, atraso, estado, categoria
- **rotulo**: satisfeito = nota >= 4

In [ ]:
dados = pd.DataFrame({
    "texto": df["review_comment_message"].fillna("").str.lower().str.replace(r"[^a-zà-ú0-9 ]", "", regex=True),
    "preco": df["price"],
    "frete": df["freight_value"],
    "dias_entrega": df["delivery_days"],
    "dias_estimado": df["estimated_delivery_days"],
    "atrasou": df["delivered_late"].astype("Int64").fillna(0),
    "estado": df["customer_state"],
    "categoria": df["product_category_en"].fillna("desconhecida"),
})
dados["satisfeito"] = (df["review_score"] >= 4).astype(int)
dados["tem_texto"] = (dados["texto"].str.len() > 0).astype(int)
dados = dados.dropna(subset=["dias_entrega"])
dados.head()

## 3. Esconder uma parte para testar (holdout)

Separamos 20% que o modelo nunca vera no treino — e assim que sabemos se ele
aprendeu de verdade. `random_state=42` deixa o resultado reproduzivel.

In [ ]:
from sklearn.model_selection import train_test_split

X = dados.drop(columns=["satisfeito"])
y = dados["satisfeito"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Treino: {len(X_tr):,}  |  Teste: {len(X_te):,}")

## 4. Montar o modelo combinado

Usamos um `ColumnTransformer`: ele aplica **TF-IDF** na coluna de texto e padroniza
as colunas numericas e categoricas — tudo de uma vez. Depois, a Regressao Logistica
aprende com todas essas informacoes juntas.

Quando o texto e vazio, o TF-IDF vira so zeros e o modelo se apoia na tabela.
**Sem `if`, sem escolher** — o modelo lida com isso sozinho.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

NUM = ["preco", "frete", "dias_entrega", "dias_estimado", "atrasou"]
CAT = ["estado", "categoria"]

preproc = ColumnTransformer([
    ("texto", TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=3), "texto"),
    ("num",   Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
    ("cat",   OneHotEncoder(handle_unknown="ignore", min_frequency=30), CAT),
])

modelo = Pipeline([
    ("features", preproc),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
modelo.fit(X_tr, y_tr)
print("Modelo combinado treinado!")

## 5. Avaliar — e provar que cobre todo mundo

O segredo esta em olhar a acuracia separando quem **tem** e quem **nao tem** texto.

In [ ]:
from sklearn.metrics import accuracy_score

pred = modelo.predict(X_te)
tem_te = X_te["tem_texto"].values

print(f"Acuracia GERAL (todos):     {accuracy_score(y_te, pred)*100:.1f}%")
print(f"  - pedidos COM texto:      {accuracy_score(y_te[tem_te==1], pred[tem_te==1])*100:.1f}%")
print(f"  - pedidos SEM texto:      {accuracy_score(y_te[tem_te==0], pred[tem_te==0])*100:.1f}%")

**Repare:** o modelo acerta bem tanto em quem escreveu quanto em quem nao escreveu.
Ele usa o texto como bonus quando existe, e se apoia na tabela quando nao existe.

Compare com a alternativa:
- modelo **so de texto**: otimo (~89%), mas cego para 60% dos pedidos
- modelo **so de tabela**: cobre todos, mas mais fraco (~73%)
- modelo **combinado**: cobre todos **e** aproveita o texto (~86%)

A licao: **nao escolha entre texto e tabela — use os dois.**

## 6. Testar com exemplos

In [ ]:
def prever(texto, preco, frete, dias_entrega, dias_estimado, atrasou, estado, categoria):
    linha = pd.DataFrame([{
        "texto": texto.lower(), "preco": preco, "frete": frete,
        "dias_entrega": dias_entrega, "dias_estimado": dias_estimado,
        "atrasou": atrasou, "estado": estado, "categoria": categoria, "tem_texto": int(len(texto) > 0),
    }])
    p = modelo.predict_proba(linha)[0][1]
    print(f"{'POS' if p>=0.5 else 'NEG'} ({p:.0%})  texto='{texto}' | atraso={atrasou}")

# com texto
prever("produto excelente, recomendo!", 120, 15, 8, 12, 0, "SP", "electronics")
# SEM texto, mas entregou MUITO atrasado -> o modelo usa a tabela
prever("", 300, 50, 40, 15, 1, "BA", "furniture")

## 7. Salvar o modelo e as predicoes

Fecha o ciclo: **coletar -> tratar -> treinar -> prever -> salvar**.

In [ ]:
import joblib

joblib.dump(modelo, "modelo_combinado.joblib")

resultado = X_te.copy()
resultado["satisfeito_real"] = y_te.values
resultado["previsao"] = pred
resultado.to_parquet("predicoes_combinado.parquet")

print("Salvos: modelo_combinado.joblib + predicoes_combinado.parquet")

## Recapitulando

- Um **unico modelo** usa texto **e** tabela.
- Quando falta texto, ele se apoia na tabela -> cobre **100%** dos pedidos.
- Texto entra como bonus quando existe.

**Na Aula 4:** um agente de IA (LLM) vai classificar os reviews **sem treinar nada** —
so lendo. Sera que ele bate este modelo combinado?